# Homework #2

By: Jenny Park\
Date: April 7, 2026\
Class: QMSS GR5069

In [0]:
from pyspark.sql.functions import floor, datediff, current_date, col, avg, min, max, rank
from pyspark.sql import Window
from pyspark.sql import functions as F

## Question 1
What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

#### Question Logic
First, I read in the dataset containing times for pit stops ("pit_stops.csv"). Then, I display to understand the contents of the table. I can see that for each race and driver, the duration along with the milliseconds of each pitstop is available. 

To figure out what the average time each driver spent at the pit stop for each race was, I grouped by raceID and driverID to get one entry for each combination of raceID and driverID, and aggregated the rows by taking the average of the "milliseconds" column of the dataset. I displayed the dataset to see the results. 

Finally, to get the slowest and fastest pit stop in each race, I grouped only by raceID this time and aggregated the rows by taking the min and max of the "milliseconds" column of the dataset. I stored these in two separate tables and displayed them. 

In [0]:
# reads in pitstop dataset and displays
df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv', header=True)
                         
display(df_pitstops)

In [0]:
# What was the average time each driver spent at the pit stop for each race?

# group by raceId and driverId and average the duration
df_pitstop_avgs = df_pitstops.groupBy('raceId', 'driverId').agg(avg('milliseconds'))

display(df_pitstop_avgs)

In [0]:
# Provide also the slowest and fastest pit stop in each race.

# groups by 'raceId' and takes the minimum/maximum milliseconds per race
df_slowest_pitstops = df_pitstops.groupBy('raceId').agg(min('milliseconds'))
df_fastest_pitstops = df_pitstops.groupBy('raceId').agg(max('milliseconds'))

display(df_slowest_pitstops)
display(df_fastest_pitstops)

#### What the Code Does
Cell 5 reads in the "pit_stops.csv" dataset, accepting the first row in the file as the header rows. Then, the 'display()' command outputs the contents of the dataframe. 

Cell 6 takes the df_pitstops dataset and groups by 'raceID' and 'driverID' to collapse rows with the same combination of 'raceID' and 'driverID' and aggregates the 'milliseconds' column by taking the average. Then, it stores it into a new dataframe called 'df_pitstop_avgs', and the 'display()' command outputs the contents of the dataframe.

Cell 7 further collapses rows by grouping by 'raceID' only in order to get the min and the max number of milliseconds for each race. The minimums are stored in a dataframe called 'df_slowest_pitstops' and the maximums are stored in a dataframe called 'df_fastest_pitstops'. Then, the 'display() function displays both dataframes.

#### Alternative Routes to Answer the Question

I could have also used the "duration" column to generate the average time each driver spent at the pit stop for each race as well as the corresponding summary statistics. This approach would have required cleaning the "duration" column to reflect a standard "mm:ss:sss" timestamp format, which would have required padding the front of rows with pitstop durations less than one minute with "00:". 

Accordingly, in order to get the average duration of type timestamp, I would have had to convert the column to Unix timestamps, take the average, then cast them back into regular timestamps. For minimum and maximum durations, I would have had to use "earlest" and "latest" rather than the min/max functions.

## Question 2

Rank order by finishing position the average time spent at the pit stop in each race.

#### Question Logic

First, I knew I needed finishing position data for each driver for each race, which the pitstops data did not have. Therefore, I read in the "results.csv" dataset which contained finishing position data in the "positionOrder" column. I chose to use this column because it assigned positions to drivers who did not finish, or whose positions were '\N' in the race. I then selected the primary key columns ('raceId', 'driverId') as well as the column with the finishing position data ('positionOrder') and joined it to the dataset I had created previously in Q1 containing 'raceId', 'driverId', and average number of milliseconds stopped at the pitstop. Using this joined dataset, I was able to rank within each race by the finishing position data, which had the corresponding average time spent at the pit stop in milliseconds. 


In [0]:
# Read in 'results.csv' dataset and display
df_results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv', header=True)

display(df_results)

In [0]:
# join with df_pitstop_avgs
# chose to use 'positionOrder' because it assigned positions to drivers who did not finish the race
df_pitstop_avgs_results = df_pitstop_avgs.join(
    df_results.select('raceId', 'driverId', 'positionOrder'), 
    on=['raceId', 'driverId'], 
    how='inner')

display(df_pitstop_avgs_results)

In [0]:
# rank within each race
window_spec = Window.partitionBy('raceId').orderBy('positionOrder')

df_ranked = df_pitstop_avgs_results.withColumn('positionOrder', rank().over(window_spec))

display(df_ranked)

#### What the Code Does

In Cell 12, I read in the 'results.csv' dataset which contains the finishing position data for each driver for each race. The 'header=True' flag ensures that the first line of the dataset is being read as headers for the table. I display the results to get a better idea of what is in the dataset.

In Cell 13, I join the relevant columns (primary key and finishing position) from the results dataset to my previously created dataset containing the average pitstop durations in milliseconds for each driver for each race. I conducted an inner join to preserve only the rows that have matching ('raceId', 'driverId') values in both dataframes. I then display the results.

In Cell 14, I rank the drivers by their finishing positions within each race using the Window function. This function enables me to partition by 'raceId' and rank by 'positionOrder' within each race, rather than the entire dataset. I store the ranked results in a dataframe called 'df_ranked' and display the results. The results show that the average number of milliseconds for each driver for each race is ordered by the driver's finishing position for each race. 

#### Alternative Routes to Answer the Question

Instead of using 'positionOrder', I could have used the 'positionText' column to rank the drivers. This would have required preliminary cleaning and decision-making surrounding how to numerically represent 'R' for retired, which indicates that a driver did not finish the race due to mechanical failures or a crash. Once cleaned, the rest of the code would have looked the same. However, the end resuls might have differed as the 'positionOrder' column assigned numerical ranks even to those who did not finish the face; thus, those drivers without an official finishing position might have been ordered differently as compared to using 'positionOrder'.

## Question 3
Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

#### Question Logic

Upon looking at the drivers.csv dataset, I can see that there are some rows where the code for drivers is missing (marked by '\N' character). When looking at the format of the codes for rows in which they do exist, it is clear that the code represents the first three letters, capitalized, of drivers' surnames. Therefore, the logic to answer this question is to identify the rows in which the code is missing (or == '\N') and replace them with the capitalized first three letters of the corresponding drivers' surname. 

In [0]:
# reading in drivers.csv dataset
df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header=True)

display(df_drivers)

In [0]:
# replacing '\N' values in the 'code' column with the capitalized first three letters of drivers' last names, based on clearly established pattern of how codes were formed
df_drivers = df_drivers.withColumn(
    'code',
    F.when(
        F.col('code') == '\\N',
        F.upper(F.substring(F.col('surname'), 1, 3))
    ).otherwise(F.col('code'))
)

display(df_drivers)

#### What the Code Does

Cell 19 begins by reading in the 'drivers.csv' dataset that contains information about each driver, including 'code' and 'surname'. Marking the header flag as true allows the first row of the data file to be read in as the header. I then display the dataframe to ensure the output is as I am expecting.

Cell 20 then uses a function to identify the rows which contain '\N' in the 'code' column, and then replaces them with the capitalized first three letters of the driver's surname using the 'upper()' and 'substring()' function. I then display the dataframe to ensure the output is as I am expecting.

#### Alternative Routes to Answer the Question

Instead of using substring to pull out the first three letters of each driver's surname, I could have used regular expressions to identify the first three letters instead. While keeping much of the code the same, I would have used a regular expression '^(.{3})' and the regexp_extract() function in place of substring() to identify the first three letters.

## Question 4
Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

#### Question Logic
First, I define what I mean by 'age' in this context. I interpret 'age' to represent the difference between the current date and each driver's birth date to represent the number of years since their birth. I calculated the current age of each driver according to this definition.

Then, in order to retain race information that each driver was in, I decide to join with the already loaded 'df_results' dataframe, which contains information about each race and driver, and their corresponding results. I select only the necessary columns ('driverId' and 'age') from the 'df_drivers' dataframe and join on the 'driverId' column with 'df_results'.

Finally, I am able to group the dataframe by 'raceId' to get two separate dataframes, one with the youngest age of all the drivers in each race and another with the oldest age of all the drivers in each race. 

In [0]:
# adding a new column in df_drivers to calculate the age of each driver
df_drivers = df_drivers.withColumn('age', floor(datediff(current_date(), col('dob')) / 365))

display(df_drivers)

In [0]:
# joining the age column of df_drivers with df_results to get the races that each driver was in
df_results_ages = df_results.join(
    df_drivers.select('driverId', 'age'), 
    on='driverId', 
    how='inner'
)

display(df_results_ages)

In [0]:
# grouping by 'raceId' and finding the youngest and oldest drivers
df_youngest_drivers = df_results_ages.groupBy('raceId').agg(min('age'))
df_oldest_drivers = df_results_ages.groupBy('raceId').agg(max('age'))

display(df_youngest_drivers)
display(df_oldest_drivers)

#### What the Code Does

Cell 25 takes the 'df_drivers' dataframe (which has already been loaded) and adds a new column titled 'age' which will contain the age of each driver. The 'datediff()' function allows me to take the difference between the current date (today) and the driver's date of birth (from the 'dob' column), dividing by 365 to convert from the number of days to the number of years. The 'floor()' function allows the age to be rounded down to the nearest year. I then display the dataframe.

Cell 26 then joins the 'age' column of the 'df_drivers' dataframe with 'df_results' to get the races that each driver was in. Only the relevant columns ('driverId' and 'age') were selected from df_drivers and an inner join on 'driverId' was performed with 'df_results'. This ensures that only rows that have matching 'driverId' values will be retained in the join. I then display the dataframe. 

Cell 27 produces two new dataframes, one containing the ages of the youngest drivers in each race and the other containing the oldest. I do so by grouping by 'raceId', retaining only the min/max of the 'age' column within each race. I then display both dataframes. 

#### Alternative Routes to Answer the Question

Instead of calculating the driver's age in the current day, I could have calculated their age at the time of each race. I would have done so by using the 'date' column from the 'races.csv' dataset, and using this information rather than 'current_date()' when calculating drivers' ages. This would have changed the oldest and youngest ages for each race because it uses age at the time of the race rather than current age. The rest of the code would have remained the same. 

## Question 5
At any given race, how many podiums does each driver have? Create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver.

#### Question Logic

Since the question is asking how many podiums (first, second, or third place) each driver has at any given race, the first step is  to read in the 'races.csv' file to obtain the race dates for each race. This will enable keeping track of the number of podiums each driver has up to that race. After joining this data with 'df_results', I use the Window function again to assist me in creating three new columns that will hold the cumulative sums of the number of podiums each driver has in chronological order. The 'num_first_places' column will count how many times the driver has a 'positionOrder' of 1 up to that race, the 'num_second_places' column will count how many times the dirver has a 'positionOrder' of 2 up to that race, and vice versa for 'num_third_places'. I also use the same technique to sum the number of times the driver has not podiumed, but completed a race.

In [0]:
# read in 'races.csv' dataset to get race dates
df_races = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv', header=True)

display(df_races)

In [0]:
# join df_results with df_races to get race dates
df_results_racedates = df_results.join(
    df_races.select('raceId', col('date').alias('raceDate')), 
    on='raceId', 
    how='inner'
)

# cast raceDate column to Date
df_results_racedates = df_results_racedates.withColumn('raceDate', col('raceDate').cast('Date'))

display(df_results_racedates) 

In [0]:
# create a Window spec that groups by 'driverId', and orders by 'raceDate'
window_spec = Window.partitionBy('driverId').orderBy('raceDate').rowsBetween(Window.unboundedPreceding, Window.currentRow)

# sum the number of first, second, and third places as well as the total number of completed races (that the driver did not podium in)
df_results_podiums = df_results_racedates.withColumn(
    'num_first_places',
    F.sum(F.when(F.col('positionOrder') == 1, 1).otherwise(0)).over(window_spec)).withColumn(
    'num_second_places',
    F.sum(F.when(F.col('positionOrder') == 2, 1).otherwise(0)).over(window_spec)).withColumn(
    'num_third_places',
    F.sum(F.when(F.col('positionOrder') == 3, 1).otherwise(0)).over(window_spec)).withColumn('num_races_completed_not_won', F.sum(F.when(~F.col('position').isin(['\\N', '1', '2', '3']), 1)).over(window_spec))

# display only the relevant columns
display(df_results_podiums.select('raceId', 'driverId', 'raceDate', 'num_first_places', 'num_second_places', 'num_third_places', 'num_races_completed_not_won'))

#### What the Code Does
In cell 32, I first read in the 'races.csv' dataset that contains information about each race and the date it was held. Setting the 'header' flag to True ensures that the first line of the data file is being read in as the headers. I then display the dataset. 

In cell 33, I join the race date column from 'df_races' to the 'df_results' dataframe. I rename the column to specify 'raceDate' using the alias() function, and perform an inner join on the column 'raceId'. Then, I cast the 'raceDate' column to be of type Date to enable ordering. I then display the data.

In cell 34, I create a Window spec that orders the races chronologically by 'raceDate' while grouping by 'driverId'. For each row, I include all previous races for that driver up to and including this race when computing the function using the 'rowsBetween()' function. Applying this Window spec then allows me to cumulatively sum each time a driver's 'positionOrder' is 1, 2, or 3. I store these cumulative sums in three new columns. I also sum each time the driver's position is not 1, 2, 3, or \N, which signifies that the driver did not complete the race. I store this result in a separate column named 'num_races_completed_not_won'. I then display only the relevant columns.

#### Alternative Routes to Answer the Question

An alternative way to answer this question would have been to interpret "at any given race" to mean grouping by 'driverId' and displaying cumulative sums for the number of podiums each driver had throughout their F1 racing career. This would have required grouping by 'driverId' and aggregating all of the times their 'positionOrder' was 1, 2, or 3 in all of their races. This would have been an easier way, albeit less detailed, to answer this question. 

## Question 6
Continue exploring the data by answering your own question.

**Question:** Which driver had the highest number of podiums in one year?

#### Question Logic

Building off the previous exercise of compiling the number of podiums that each driver has, I single out the year from the race date, and then group by 'year' and 'driverId'. I then sum the number of first, second, and third places each driver got in each year, then create a new column named 'num_podiums' for each driver for each year. Then, I sort the dataset in descending order by 'num_podiums' to identify the driver that had the highest number of podiums in one year. Finally, I join the dataset with the first and last name of the driver, according to 'driverId' to identify the name of the driver.

In [0]:
# creating column 'year' from 'raceDate'
df_results_year = df_results_podiums.withColumn('year', F.year('raceDate'))

display(df_results_year)

In [0]:
df_year_podiums = df_results_year.groupBy('driverId', 'year').agg(
    F.sum('num_first_places').alias('num_first_places'),
    F.sum('num_second_places').alias('num_second_places'),
    F.sum('num_third_places').alias('num_third_places')
).withColumn(
    'num_podiums',
    F.col('num_first_places') + F.col('num_second_places') + F.col('num_third_places')
)

display(df_year_podiums)

In [0]:
highest_podium = df_year_podiums.orderBy(F.col('num_podiums').desc())

highest_podium_driver_name = df_drivers.join(
    highest_podium, 
    on='driverId', 
    how='inner'
).select('forename', 'surname').limit(1)

display(highest_podium_driver_name)


#### What the Code Does

In cell 39, I use the previously created 'df_results_podiums' dataframe and create a new column that isolates the 'year' from the 'raceDate' column. I then display the dataset. 

In cell 40, I create a new dataset that groups by 'year' and 'driverId', aggregating the number of times each driver received first, second, and third place in each year. I also create a new column that contains the total number of podiums called 'num_podiums' I then display the dataset.

In cell 41, I sort the 'df_year_podiums' dataframe in descending order of 'num_podiums' using the 'orderBy()' and 'desc()' function. I then join this dataset with the 'df_drivers' dataset on 'driverId' in order to get the first and last name of the driver with the highest number of podiums. I then display the name of the driver to answer the question. 

#### Alternative Routes to Answer the Question

Instead of limiting to one year, I could have ranked the number of podiums in a driver's entire career, aggregating across years instead of within years. This would have required changing the 'groupBy' condition to only include 'driverId' rather than both 'year' and 'driverId'. 